In [ ]:
import openpmd_api as io

In [ ]:
series_path = "../diags/fields_sliced/openpmd.bp5/"

In [ ]:
s = io.Series("../diags/fields/openpmd_%T.bp5/", access=io.Access_Type.read_only)
it = s.iterations[1408]
print(it.attributes)
print(list(it.meshes))
print(list(it.meshes['E']))

In [ ]:
from warpx_openpmd_movie_v6 import (
    read_field_linear_capped,
    plot_warpx_slice_imshow,
    save_series_frames_linear,
)

In [ ]:
arr, meta = read_field_linear_capped(
    series_path,
    target_iteration=14928,
    field="|E|",          # also: "E_mag", "abs(E)", "E/abs"
    max_linear_steps=1245,
    verbose=False,
)

print(meta["display_name"])
print(meta["derived_from"])
print(arr.shape)

In [ ]:
fig, ax, cax, im = plot_warpx_slice_imshow(
    arr,
    meta,
    slice_axis=0,
    slice_reduction="mean",
    autoscale_percentile=99.5,
    colorbar_unit="V/m",
)

In [ ]:
outdir = "movie_frames_Emag"

n_saved = save_series_frames_linear(
    series_path,
    field="|E|",                 # also works: "E_mag", "abs(E)", "E/abs"
    outdir=outdir,
    max_linear_steps=1245,       # your safety cap
    every=1,
    slice_axis=0,                # for shape like (z, y, x)
    slice_reduction="mean",      # or "first"
    autoscale_percentile=99.5,   # use None for exact max
    colorbar_unit="V/m",
    verbose=True,
    cmap="viridis"
)

print(f"Saved {n_saved} frames")

In [ ]:
import subprocess

cmd = [
    "ffmpeg",
    "-framerate", "30",
    "-pattern_type", "glob",
    "-i", "movie_frames_Emag/*.png",
    "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2,format=yuv420p",
    "-c:v", "libx264",
    "-preset", "medium",
    "-crf", "18",
    "Emag_movie.mp4",
]

subprocess.run(cmd, check=True)